In [2]:
import pandas as pd
from urllib.parse import urlparse  # For hostname extraction

# Load the dataset
df = pd.read_csv('phishing_site_urls.csv')  # Assuming the filename is "phishing_site_urls.csv"

print(df.head())

print(df.isnull().sum())

print(df['Label'].value_counts())  # Using 'label' instead of 'type'


                                                 URL Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...   bad
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...   bad
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....   bad
3  mail.printakid.com/www.online.americanexpress....   bad
4  thewhiskeydregs.com/wp-content/themes/widescre...   bad
URL      0
Label    0
dtype: int64
Label
good    392924
bad     156422
Name: count, dtype: int64


In [4]:
import re
import tldextract
from urllib.parse import urlparse

def extract_features(url):
    features = {}
    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    features['url_length'] = len(url)
    features['num_dots'] = url.count('.')
    features['num_hyphens'] = url.count('-')
    features['num_underscores'] = url.count('_')
    features['num_slashes'] = url.count('/')
    features['num_percent'] = url.count('%')
    features['num_question_marks'] = url.count('?')
    features['num_equals'] = url.count('=')
    features['num_at'] = url.count('@')
    features['has_ip'] = int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', parsed.netloc)))
    features['has_https'] = int(parsed.scheme == 'https')
    features['domain_length'] = len(ext.domain)
    features['num_subdomains'] = len(ext.subdomain.split('.')) if ext.subdomain else 0
    features['path_length'] = len(parsed.path)
    
    return features

# Apply feature extraction to the dataset
feature_list = df['URL'].apply(extract_features)
features_df = pd.DataFrame(feature_list.tolist())

# Combine the features with the original labels
df_features = pd.concat([features_df, df['Label']], axis=1)

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_features['label'] = le.fit_transform(df_features['Label'])
df_features.drop('Label', axis=1, inplace=True)

In [8]:
print(df_features.head())

   url_length  num_dots  num_hyphens  num_underscores  num_slashes  \
0         225         6            4                4           10   
1          81         5            2                1            4   
2         177         7            1                0           11   
3          60         6            0                0            2   
4         116         1            1                0           10   

   num_percent  num_question_marks  num_equals  num_at  has_ip  has_https  \
0            0                   1           4       0       0          0   
1            0                   0           2       0       0          0   
2            0                   0           0       0       0          0   
3            0                   0           0       0       0          0   
4            0                   1           0       0       0          0   

   domain_length  num_subdomains  path_length  label  
0              6               0          134      0  
1     

In [9]:
df_features.to_csv('lexical_features.csv', index=False)